# Figure 3 — six-colour quantum-dot sorting (Ximea)

**Makes up:** Figure 3 e (max projection of one field of view, each QDot circled with its assigned
identity), f (ternary plot of all detected QDots, coloured by assigned identity) and g (confusion
matrix of a synthetic dataset of 4,500 of each QDot). Also the mixture's ternary with each
single-QDot population outlined, and two SI panels from the same data:
`QuantumDots/QDotConfusion_True_vs_Sorted` and `Limitations/Qdot585_Qdot655_GvsR`.

Figure 3 a–c (dye pairs and their misidentification rate) come from a different notebook.

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/QDots/`: the mixture
`MixB/1nM525_200pM585_200pM605_50pM655_200pM705_50pM800_…_30p515_1/` and six single-QDot
references (`QDot525/585/605`: the 30 % 515 nm acquisition; `QDot655/700/800`: the 10 % and 30 %
acquisitions pooled). Each acquisition is a 5 × 5 grid of positions, one TIFF (500 frames) and one
MicroManager metadata file each. The deposition holds raw data only, so this notebook fits it first.

**Method** (from the paper's analysis): fit each position → AIM drift correction per position →
HDBSCAN clustering of localisations into molecules → keep mixture molecules seen in more than 20
frames → assign each to a QDot with a Gaussian mixture whose means and covariances are fixed to
the single-QDot references (only the mixing weights are fitted). The shared code is in
`_qdots.py`, also used by `SI/FigureSI_QDotConfusion.ipynb`.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits one position per acquisition: the mixture's
pictured position (`Pos001_001`) and the first position of each of the nine reference
acquisitions (~4 GB, roughly 1 h of fitting). Enough to run everything end to end, with ~1/25 of the
molecules, so the reference outlines are rougher than in the paper.
`FAST_MODE = False` uses all 250 positions (~75 GB, many hours of fitting; run in `tmux`/`screen`).

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py and _qdots.py live beside this notebook (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve()))
import _qdots as qd

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import SM_extractionfunctions
from pyS3M import PlottingBase
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
SM_E = SM_extractionfunctions.extract_SMs()
drift_corrector = Drift_Correction_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: one position per acquisition (~4 GB). False: all 250 positions (~75 GB, the published data).
REFIT = False      # True: fit again even if fits with the same files and settings exist.

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_3_QDots"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_ROOT at a drive with room for the download. Each acquisition gets its own
# sub-folder, separately for each FAST_MODE setting: the fit reads every TIFF inside it.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")
print("Fit settings:", qd.FIT_PARAMS)

---
## 2 — Download, fit and drift-correct every acquisition

For each acquisition: download from BioStudies (with each position's metadata, which holds the
camera ROI the fit needs), fit every position separately (`fit_SM_data`, one `.h5` per TIFF), then
AIM drift correction per position. Re-running is safe: complete downloads are skipped, and fits
are reused unless the files or settings change (or `REFIT = True`).

In [ ]:
def prepare(remote_folder, dest_dir, position=None):
    return qd.prepare_acquisition(remote_folder, dest_dir, FAST_MODE, SupRes_F, calib,
                                  smoothing_function, drift_corrector, IO,
                                  position=position, refit=REFIT)

mix_files = prepare(qd.MIX_FOLDER, DATA_ROOT / "MixB", position=qd.MIX_PANEL_POSITION)

ref_files = {}
for name, (_, folders) in qd.REFERENCES.items():
    ref_files[name] = []
    for folder in folders:
        ref_files[name] += prepare(folder, DATA_ROOT / name / Path(folder).name)

print(f"Mixture: {len(mix_files)} position(s)")
for name, files in ref_files.items():
    print(f"{name}: {len(files)} position(s)")

---
## 3 — Cluster localisations into molecules

HDBSCAN, min. cluster size 10 localisations (as in the paper's Methods), after quality cuts
(chi² < 2, localisation error < 1 px, 100–10⁶ photons). The analysis notebook behind the published
panels re-clustered the mixture with DBSCAN instead; pass `params=qd.DBSCAN_MIX_PARAMS` to the
mixture's `cluster_sample` call to reproduce that. One molecule's colour (`A_B`, `A_G`, `A_R`: its relative signal on the three
pixel types) is the inverse-variance weighted mean over its localisations. The references' positions are
pooled per QDot.

In [ ]:
mix_sm, mix_sf = qd.cluster_sample(SM_E, mix_files, DATA_ROOT / "MixB", "MixB")
ref_sm = {name: qd.cluster_sample(SM_E, files, DATA_ROOT / name, name)[0]
          for name, files in ref_files.items()}

print(f"Mixture: {len(mix_sm):,} molecules")
for name, db in ref_sm.items():
    print(f"{name}: {len(db):,} molecules")

---
## 4 — Assign each mixture molecule to a QDot

Molecules seen in more than 20 frames only (better colour estimates). Each reference gives one
Gaussian component: its mean colour and per-channel variance (diagonal covariance). These stay
fixed; EM fits only the six mixing weights, and each molecule goes to its most likely component.

In [ ]:
MIN_FRAMES = 20

mix = mix_sm[mix_sm["frames"] > MIN_FRAMES].copy()
names = list(qd.REFERENCES)
means, covariances = qd.reference_components(ref_sm)
print(pd.DataFrame(means, index=names, columns=["A_R", "A_G", "A_B"]).round(4))

model = qd.FixedParamsGMM(means, covariances).fit(mix[["A_R", "A_G", "A_B"]].to_numpy())
mix["qdot"] = np.array(names)[model.labels_]

summary = pd.DataFrame({"fitted weight": model.weights_,
                        "molecules": mix["qdot"].value_counts().reindex(names, fill_value=0)},
                       index=names)
print(f"\n{len(mix):,} molecules with > {MIN_FRAMES} frames ({model.n_iter_} EM iterations)")
print(summary.round(3))

---
## 5 — Ternary plot of the mixture with the single-QDot outlines

Density of the mixture's molecules; each reference QDot's 50 % density contour on top. The blue
fraction is stretched ×6 (then renormalised) to separate the QDots, which all have little blue signal.

In [ ]:
BLUE_SCALE = 6

fig, ax = plotter.create_ternary_density(
    mix["A_R"], mix["A_G"], BLUE_SCALE * mix["A_B"], gridsize=75, cmap="gray", vmin=10,
    log_scale=True, show_colorbar=False, grid_spacing=0.2,
)
for name, (colour, _) in qd.REFERENCES.items():
    ref = ref_sm[name]
    plotter.plot_ternary_kde_contours(ax, ref["A_R"], ref["A_G"], BLUE_SCALE * ref["A_B"],
                                      levels=[0.5], color=colour, alpha=0.5)
ax.grid(lw=1, alpha=0.25, ls="--", which="both", axis="both", color="white")
plt.savefig(FIG_DIR / "QuantumDotTernary_6timesBlue.svg", dpi=600, format="svg")
plt.show()

---
## 6 — Figure 3f: the same molecules, coloured by their assigned QDot

Point opacity is scaled to each QDot's number of molecules, so sparse and abundant QDots show alike.

In [ ]:
fig, ax = plotter.create_ternary_density(
    mix["A_R"], mix["A_G"], BLUE_SCALE * mix["A_B"], gridsize=100, cmap="gray", vmin=10,
    log_scale=True, show_colorbar=False, grid_spacing=0.2,
)
for name, (colour, _) in qd.REFERENCES.items():
    sub = mix[mix["qdot"] == name]
    if len(sub):
        ax.scatter(sub["A_R"], sub["A_G"], BLUE_SCALE * sub["A_B"], color=colour, s=1,
                   alpha=min(1.0, 40 / len(sub)), rasterized=True)
plt.savefig(FIG_DIR / "QuantumDotTernary_6timesBlue_SortedPoints.svg", dpi=600, format="svg")
plt.show()

---
## 7 — Figure 3e: one field of view with the assigned QDots circled

Maximum projection of the raw frames of position `Pos001_001`, with each assigned molecule
circled in its QDot's colour. The crop is in camera pixels of the acquisition's ROI.

In [ ]:
CROP_X, CROP_Y = (465, 535), (450, 550)
CIRCLE_COLOURS = {"QDot525": "#9a589a", "QDot585": "blue", "QDot605": "green",
                  "QDot655": "gold", "QDot705": "darkorange", "QDot800": "red"}

panel_fov = next(i for i, f in enumerate(mix_files) if f"-{qd.MIX_PANEL_POSITION}" in f.name)
panel_tif = mix_files[panel_fov].with_name(
    mix_files[panel_fov].name.replace("_undrifted_locs.h5", ".ome.tif"))
max_projection = np.max(IO.read_tiff(panel_tif), axis=0)

fig, ax = plotter.two_column_plot(nrows=1, ncols=1, height=1.885, width=1.337)
plotter.image_plot(ax, max_projection, vmin=60, vmax=150, scalebar=True, pixelsize=qd.PIXEL_SIZE_NM,
                   scalebarsize=1000, scalebarlabel="1 μm")
in_fov = mix[mix["fov_index"] == panel_fov]
for name, colour in CIRCLE_COLOURS.items():
    sub = in_fov[in_fov["qdot"] == name]
    plotter.scatter_plot(ax, sub["xc"], sub["yc"], xlabel="", ylabel="", linewidth=1, size=100,
                         edgecolor=colour, facecolor="none", alpha=1, grid=False)
ax.set_xlim(CROP_X)
ax.set_ylim(CROP_Y)
plt.savefig(FIG_DIR / "QuantumDotAssignedRawImage_MIP.svg", dpi=600, format="svg")
plt.show()

in_crop = in_fov[in_fov["xc"].between(*CROP_X) & in_fov["yc"].between(*CROP_Y)]
print(f"{len(in_crop)} assigned molecules in the crop:")
print(in_crop["qdot"].value_counts().reindex(names, fill_value=0))

---
## 8 — Figure 3g: confusion matrix of a synthetic six-QDot dataset

An even synthetic mixture: 4,500 molecules drawn at random from each single-QDot reference (fewer in
`FAST_MODE` if a reference has fewer), sorted with the same fixed-component mixture model.
Each row is one true QDot; the entries are the fraction assigned to each QDot.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

N_PER_QDOT = 4500
rng = np.random.default_rng(0)

n_draw = min(N_PER_QDOT, min(len(db) for db in ref_sm.values()))
if n_draw < N_PER_QDOT:
    print(f"Only {n_draw} molecules in the smallest reference: drawing {n_draw} per QDot "
          f"(the paper used {N_PER_QDOT})")
even_mix = pd.concat(
    [ref_sm[name].iloc[rng.choice(len(ref_sm[name]), n_draw, replace=False)].assign(true_qdot=name)
     for name in names],
    ignore_index=True,
)

even_model = qd.FixedParamsGMM(means, covariances).fit(even_mix[["A_R", "A_G", "A_B"]].to_numpy())
even_mix["qdot"] = np.array(names)[even_model.labels_]

cm = confusion_matrix(even_mix["true_qdot"], even_mix["qdot"], labels=names, normalize="true")
ConfusionMatrixDisplay(cm, display_labels=names).plot(cmap=plt.cm.gist_yarg, values_format=".2f")
plt.savefig(FIG_DIR / "QDot_Confusion_Matrix_Even.svg", dpi=600, format="svg")
plt.show()

print(f"Correctly assigned: {np.mean(even_mix['qdot'] == even_mix['true_qdot']):.3f} "
      f"({len(even_mix):,} molecules)")

---
## 9 — SI: the synthetic dataset, true vs sorted identity

Left: each molecule coloured by the QDot it came from; right: by the QDot it was assigned to.

In [ ]:
for tag, column in [("GroundTruth", "true_qdot"), ("Sorted", "qdot")]:
    fig, ax = plotter.create_ternary_density(
        even_mix["A_R"], even_mix["A_G"], BLUE_SCALE * even_mix["A_B"], gridsize=100, cmap="gray",
        vmax=150, show_colorbar=False,
    )
    for name, (colour, _) in qd.REFERENCES.items():
        sub = even_mix[even_mix[column] == name]
        ax.scatter(sub["A_R"], sub["A_G"], BLUE_SCALE * sub["A_B"], color=colour, s=1, alpha=0.03)
    plt.savefig(FIG_DIR / f"QDot_Confusion_Tern_{tag}.svg", dpi=600, format="svg")
    plt.show()

---
## 10 — SI: Qdot 585 vs Qdot 655 (pixel 2 vs pixel 3 QE)

Two QDots whose colours lie close together on the ternary plot: 2D histograms of their molecules'
relative pixel 2 (`A_G`) and pixel 3 (`A_R`) QE.

In [ ]:
bins = np.linspace(0, 1, 201)
fig, axs = plotter.two_column_plot(nrows=1, ncols=2)
for ax, name in zip(axs, ["QDot585", "QDot655"]):
    ax.hist2d(ref_sm[name]["A_G"], ref_sm[name]["A_R"], bins, cmap="gist_yarg", rasterized=True)
    ax.set_title(name)
    ax.set_xlabel("pixel 2 QE")
axs[0].set_ylabel("pixel 3 QE")
plt.savefig(FIG_DIR / "QDot_585_vs_655_GvsR.svg", dpi=600, format="svg")
plt.show()